In [ ]:
#@title Setup (run me first!) { display-mode: "form" }
#@markdown This cell loads the shared infrastructure, generates data, and
#@markdown precomputes all interactive elements. Takes ~20 seconds.

import json, os, warnings
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import durbin_watson, jarque_bera
from statsmodels.stats.outliers_influence import variance_inflation_factor, OLSInfluence
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from sklearn.model_selection import cross_val_score, KFold
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
warnings.filterwarnings('ignore')

# ── Color palette ──
COLORS = {
    'ols': '#2E5EA8',
    'truth': '#D4A017',
    'bias': '#C0392B',
    'repair': '#27AE60',
    'alt': '#7F8C8D',
}

# ── Gate System ──
_trap_responses = {}
_TRAP_LOG_PATH = '/content/trap_log.json'

def _load_trap_log():
    global _trap_responses
    if os.path.exists(_TRAP_LOG_PATH):
        try:
            with open(_TRAP_LOG_PATH, 'r') as f:
                _trap_responses = json.load(f)
        except (json.JSONDecodeError, IOError):
            _trap_responses = {}

def _save_trap_log():
    try:
        os.makedirs(os.path.dirname(_TRAP_LOG_PATH), exist_ok=True)
        with open(_TRAP_LOG_PATH, 'w') as f:
            json.dump(_trap_responses, f, indent=2)
    except IOError:
        pass

def record_trap_response(notebook_id, question_id, response):
    key = f"{notebook_id}_{question_id}"
    _trap_responses[key] = {"response": response, "timestamp": datetime.now().isoformat()}
    _save_trap_log()

def get_trap_response(notebook_id, question_id):
    key = f"{notebook_id}_{question_id}"
    entry = _trap_responses.get(key)
    return entry["response"] if entry else None

def check_gate(notebook_id, question_id):
    return f"{notebook_id}_{question_id}" in _trap_responses

def get_all_responses():
    return dict(_trap_responses)

def create_trap_widget(notebook_id, question_id, question_text, options):
    label = widgets.HTML(f"<b>{question_text}</b>")
    radio = widgets.RadioButtons(options=options, value=None, layout=widgets.Layout(width='auto'))
    submit = widgets.Button(description="Submit", button_style='primary')
    output = widgets.Output()
    def on_submit(btn):
        with output:
            output.clear_output()
            if radio.value is None:
                print("Please select an option before submitting.")
                return
            record_trap_response(notebook_id, question_id, radio.value)
            print(f"Response recorded: {radio.value}")
            submit.disabled = True
            radio.disabled = True
    submit.on_click(on_submit)
    existing = get_trap_response(notebook_id, question_id)
    if existing is not None:
        try:
            radio.value = existing
            radio.disabled = True
            submit.disabled = True
        except Exception:
            pass
    return widgets.VBox([label, radio, submit, output])

_load_trap_log()

# ── CompanySimulator ──
class CompanySimulator:
    def __init__(self, endogeneity_strength=20, heteroscedasticity_strength=0.5,
                 nonlinearity=True, bad_control_strength=0.1, noise_sigma=1.0):
        self.endogeneity_strength = endogeneity_strength
        self.heteroscedasticity_strength = heteroscedasticity_strength
        self.nonlinearity = nonlinearity
        self.bad_control_strength = bad_control_strength
        self.noise_sigma = noise_sigma
        self.beta_0 = 50
        self.beta_1 = 8
        self.beta_2 = 3
        self.beta_U = 2
        self.staff_loading = 5

    def generate(self, n=500, seed=42):
        rng = np.random.default_rng(seed)
        U = rng.standard_normal(n)
        eta1 = rng.normal(0, self.noise_sigma, n)
        eta2 = rng.normal(0, self.noise_sigma, n)
        eta3 = rng.normal(0, self.noise_sigma, n)
        X1 = self.endogeneity_strength * U + eta1
        if self.nonlinearity:
            X1 = np.abs(X1) + 0.01
        X2 = self.staff_loading * U + eta2
        eps_std = np.sqrt(np.abs(self.heteroscedasticity_strength * X1))
        eps = rng.normal(0, eps_std)
        if self.nonlinearity:
            Y = self.beta_0 + self.beta_1 * np.log(X1) + self.beta_2 * X2 + self.beta_U * U + eps
        else:
            Y = self.beta_0 + self.beta_1 * X1 + self.beta_2 * X2 + self.beta_U * U + eps
        X3 = self.bad_control_strength * Y + eta3
        return pd.DataFrame({'revenue': Y, 'ad_spend': X1, 'staff_count': X2, 'satisfaction': X3})

    def truth(self, n=500, seed=42):
        rng = np.random.default_rng(seed)
        U = rng.standard_normal(n)
        eta1 = rng.normal(0, self.noise_sigma, n)
        eta2 = rng.normal(0, self.noise_sigma, n)
        eta3 = rng.normal(0, self.noise_sigma, n)
        X1 = self.endogeneity_strength * U + eta1
        if self.nonlinearity:
            X1 = np.abs(X1) + 0.01
        X2 = self.staff_loading * U + eta2
        eps_std = np.sqrt(np.abs(self.heteroscedasticity_strength * X1))
        eps = rng.normal(0, eps_std)
        if self.nonlinearity:
            Y = self.beta_0 + self.beta_1 * np.log(X1) + self.beta_2 * X2 + self.beta_U * U + eps
        else:
            Y = self.beta_0 + self.beta_1 * X1 + self.beta_2 * X2 + self.beta_U * U + eps
        X3 = self.bad_control_strength * Y + eta3
        df = pd.DataFrame({'revenue': Y, 'ad_spend': X1, 'staff_count': X2,
                           'satisfaction': X3, 'demand_U': U})
        params = {'beta_0': self.beta_0, 'beta_1': self.beta_1, 'beta_2': self.beta_2,
                  'beta_U': self.beta_U, 'sigma_epsilon': self.heteroscedasticity_strength}
        return df, params

    def dgp_summary(self):
        func = r"\log(X_1)" if self.nonlinearity else r"X_1"
        return (rf"$Y = {self.beta_0} + {self.beta_1} \cdot {func} "
                rf"+ {self.beta_2} \cdot X_2 + {self.beta_U} \cdot U + \varepsilon$")

    def set_heteroscedasticity(self, strength):
        self.heteroscedasticity_strength = strength
    def set_endogeneity(self, strength):
        self.endogeneity_strength = strength
    def set_nonlinearity(self, curvature):
        self.nonlinearity = bool(curvature)

# ── MonteCarloEngine ──
class MonteCarloEngine:
    def run(self, estimator_fn, param_name, param_grid, simulator, n_reps=5000, n_obs=500):
        results = np.empty((len(param_grid), n_reps))
        try:
            progress = widgets.IntProgress(value=0, min=0, max=len(param_grid),
                                           description='Simulating:', style={'description_width': 'initial'})
            display(progress)
            use_widget = True
        except Exception:
            use_widget = False
        setter = getattr(simulator, f'set_{param_name}', None)
        for i, val in enumerate(param_grid):
            if setter is not None:
                setter(val)
            for r in range(n_reps):
                data = simulator.generate(n=n_obs, seed=r)
                results[i, r] = estimator_fn(data)
            if use_widget:
                progress.value = i + 1
        if use_widget:
            progress.bar_style = 'success'
        return results

    def quick_run(self, estimator_fn, dgp_fn, n_reps=1000, n_obs=500):
        results = np.empty(n_reps)
        for r in range(n_reps):
            data = dgp_fn(seed=r, n=n_obs)
            results[r] = estimator_fn(data)
        return results

# ── AutopsyReport ──
class AutopsyReport:
    @staticmethod
    def lightweight(notebook_number):
        threat = widgets.Text(description='Biggest threat:', placeholder='What is the biggest threat to this estimate?',
                              layout=widgets.Layout(width='90%'), style={'description_width': '120px'})
        check = widgets.Text(description='How to check:', placeholder='How would you check if that threat is real?',
                             layout=widgets.Layout(width='90%'), style={'description_width': '120px'})
        submit = widgets.Button(description='Save Autopsy', button_style='primary')
        output = widgets.Output()
        def on_submit(btn):
            with output:
                output.clear_output()
                nb_id = f"notebook_{notebook_number}"
                record_trap_response(nb_id, "threat", threat.value)
                record_trap_response(nb_id, "check", check.value)
                print("Autopsy responses saved.")
                submit.disabled = True
        submit.on_click(on_submit)
        return widgets.VBox([widgets.HTML(f"<h3>Mini Autopsy \u2014 Notebook {notebook_number}</h3>"),
                             threat, check, submit, output])

# ── Generate data ──
sim = CompanySimulator()
df = sim.generate(n=500, seed=42)

# Train/test split (fixed seed for reproducibility)
rng_split = np.random.default_rng(123)
indices = rng_split.permutation(len(df))
train_idx = indices[:350]
test_idx = indices[350:]
df_train = df.iloc[train_idx].copy()
df_test = df.iloc[test_idx].copy()

# Precompute polynomial fits for degrees 1-15
print("Precomputing polynomial models...")
progress = widgets.IntProgress(value=0, min=0, max=15, description='Fitting:', style={'description_width': 'initial'})
display(progress)

poly_results = {}
X_train = df_train[['ad_spend']].values
y_train = df_train['revenue'].values
X_test = df_test[['ad_spend']].values
y_test = df_test['revenue'].values

kf = KFold(n_splits=5, shuffle=True, random_state=42)

for deg in range(1, 16):
    pipe = make_pipeline(PolynomialFeatures(deg, include_bias=False), LinearRegression())
    pipe.fit(X_train, y_train)
    train_pred = pipe.predict(X_train)
    test_pred = pipe.predict(X_test)

    ss_res_train = np.sum((y_train - train_pred)**2)
    ss_tot_train = np.sum((y_train - y_train.mean())**2)
    r2_train = 1 - ss_res_train / ss_tot_train

    ss_res_test = np.sum((y_test - test_pred)**2)
    ss_tot_test = np.sum((y_test - y_test.mean())**2)
    r2_test = 1 - ss_res_test / ss_tot_test

    # Cross-validation on training set
    cv_pipe = make_pipeline(PolynomialFeatures(deg, include_bias=False), LinearRegression())
    cv_scores = cross_val_score(cv_pipe, X_train, y_train, cv=kf, scoring='r2')
    cv_r2 = cv_scores.mean()

    poly_results[deg] = {
        'r2_train': r2_train,
        'r2_test': r2_test,
        'cv_r2': cv_r2,
        'model': pipe,
        'train_mse': ss_res_train / len(y_train),
        'test_mse': ss_res_test / len(y_test)
    }
    progress.value = deg

# Also compute log-linear model
X_train_log = sm.add_constant(np.log(df_train['ad_spend']))
X_test_log = sm.add_constant(np.log(df_test['ad_spend']))
model_log = sm.OLS(y_train, X_train_log).fit()
log_train_pred = model_log.predict(X_train_log)
log_test_pred = model_log.predict(X_test_log)
r2_log_train = 1 - np.sum((y_train - log_train_pred)**2) / np.sum((y_train - y_train.mean())**2)
r2_log_test = 1 - np.sum((y_test - log_test_pred)**2) / np.sum((y_test - y_test.mean())**2)

progress.bar_style = 'success'
print("Setup complete!")
print(f"Training set: {len(df_train)} observations | Test set: {len(df_test)} observations")

# Notebook 5: Why Your R² Is Lying

*A model that fits your existing data perfectly will predict new data terribly. This is not a paradox — it's memorization vs. learning.*

---

You learned in Notebook 4 that the relationship between ad spend and revenue is logarithmic, not linear. So now you're trying to find the best model. You decide to try polynomial models of increasing complexity. More flexibility should mean better fit, right?

Let's build a **leaderboard** and find the winner.

In [ ]:
# ── The Trap: Polynomial leaderboard ──
# R² keeps climbing. Which model is best?

print("\u2550" * 50)
print("   MODEL LEADERBOARD \u2014 Ranked by R\u00b2")
print("\u2550" * 50)
print(f"{'Rank':<6}{'Model':<25}{'R\u00b2':>8}")
print("\u2500" * 50)

# Sort by training R2 descending
sorted_models = sorted(poly_results.items(), key=lambda x: x[1]['r2_train'], reverse=True)

for rank, (deg, res) in enumerate(sorted_models[:8], 1):
    emoji = '\u2b50' if rank == 1 else '  '
    print(f"{rank:<6}Polynomial degree {deg:<8}{res['r2_train']:.4f}  {emoji}")

print("\u2500" * 50)
print(f"\nTop model: Degree {sorted_models[0][0]} with R\u00b2 = {sorted_models[0][1]['r2_train']:.4f}")
print(f"Baseline: Degree 1 (linear) with R\u00b2 = {poly_results[1]['r2_train']:.4f}")
print(f"\nR\u00b2 progression: ", end='')
for deg in [1, 3, 7, 12, 15]:
    print(f"{poly_results[deg]['r2_train']:.2f}", end=' \u2192 ' if deg < 15 else '\n')

In [ ]:
# ── The Trap: Pick the winner ──

trap_widget = create_trap_widget(
    'notebook_5', 'best_model',
    'Based on the leaderboard above, which model would you recommend to the VP?',
    [
        f'Degree 15 (R\u00b2={poly_results[15]["r2_train"]:.3f}) \u2014 highest R\u00b2, best fit',
        f'Degree 7 (R\u00b2={poly_results[7]["r2_train"]:.3f}) \u2014 good balance of fit and simplicity',
        f'Degree 3 (R\u00b2={poly_results[3]["r2_train"]:.3f}) \u2014 modest improvement over linear',
        f'Degree 1 (R\u00b2={poly_results[1]["r2_train"]:.3f}) \u2014 simple linear model',
    ]
)
display(trap_widget)

In [ ]:
# ── The Reveal: Test set performance ──

if not check_gate('notebook_5', 'best_model'):
    print("\u26a0\ufe0f Please submit your choice in the cell above before continuing.")
else:
    student_choice = get_trap_response('notebook_5', 'best_model')

    print("\u2550" * 60)
    print("   REAL LEADERBOARD \u2014 Ranked by TEST R\u00b2")
    print("\u2550" * 60)
    print(f"{'Rank':<6}{'Model':<22}{'Train R\u00b2':>10}{'Test R\u00b2':>10}")
    print("\u2500" * 60)

    # Sort by TEST R2
    sorted_test = sorted(poly_results.items(), key=lambda x: x[1]['r2_test'], reverse=True)

    for rank, (deg, res) in enumerate(sorted_test, 1):
        marker = ''
        if res['r2_test'] < 0:
            marker = ' \u2620\ufe0f WORSE THAN MEAN'
        elif rank == 1:
            marker = ' \u2b50 ACTUAL BEST'
        print(f"{rank:<6}Degree {deg:<15}{res['r2_train']:>10.4f}{res['r2_test']:>10.4f}{marker}")

    print("\u2500" * 60)

    # Show the reversal
    worst_test = sorted_test[-1]
    best_test = sorted_test[0]
    print(f"\nThe degree-{sorted_models[0][0]} model (highest training R\u00b2) "
          f"has test R\u00b2 = {poly_results[sorted_models[0][0]]['r2_test']:.4f}")
    if poly_results[sorted_models[0][0]]['r2_test'] < 0:
        print("It predicts WORSE than just guessing the mean!")
    print(f"\nActual best: Degree {best_test[0]} (test R\u00b2 = {best_test[1]['r2_test']:.4f})")

    if 'Degree 15' in student_choice or 'highest' in student_choice:
        print("\n\u274c You picked the highest training R\u00b2. On new data, it's the worst.")
        print("The model memorized the training data's noise, not its signal.")
    elif 'Degree 1' in student_choice:
        print("\n\u2714\ufe0f Conservative choice! But we can do better with the right model family.")
    else:
        print("\nR\u00b2 on training data is not a reliable guide to model quality.")

    # Visualization
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    degrees = list(range(1, 16))
    train_r2s = [poly_results[d]['r2_train'] for d in degrees]
    test_r2s = [poly_results[d]['r2_test'] for d in degrees]

    ax1.plot(degrees, train_r2s, 'o-', color=COLORS['ols'], linewidth=2, markersize=7,
             label='Training R\u00b2')
    ax1.plot(degrees, test_r2s, 's--', color=COLORS['bias'], linewidth=2, markersize=7,
             label='Test R\u00b2')
    ax1.axhline(0, color=COLORS['alt'], linestyle=':', alpha=0.5)
    ax1.set_xlabel('Polynomial Degree', fontsize=12)
    ax1.set_ylabel('R\u00b2', fontsize=12)
    ax1.set_title('Training R\u00b2 Always Rises. Test R\u00b2 Crashes.', fontsize=13, fontweight='bold')
    ax1.legend(fontsize=11)
    ax1.set_xticks(degrees)

    # Right: fitted curves for degree 1, 3, and highest
    x_plot = np.linspace(df['ad_spend'].min(), df['ad_spend'].max(), 200).reshape(-1, 1)
    ax2.scatter(df_test['ad_spend'], df_test['revenue'], alpha=0.4, s=20,
                color=COLORS['alt'], label='Test data')

    for deg, color, ls, mk in [(1, COLORS['ols'], '-', 'o'),
                                (3, COLORS['repair'], '-.', 's'),
                                (sorted_models[0][0], COLORS['bias'], ':', '^')]:
        pred = poly_results[deg]['model'].predict(x_plot)
        ax2.plot(x_plot, pred, color=color, linewidth=2, linestyle=ls,
                 label=f'Degree {deg} (test R\u00b2={poly_results[deg]["r2_test"]:.2f})')

    ax2.set_xlabel('Ad Spend ($K)', fontsize=12)
    ax2.set_ylabel('Revenue ($K)', fontsize=12)
    ax2.set_title('High-degree polynomial: memorization, not learning', fontsize=13)
    ax2.legend(fontsize=9)

    plt.tight_layout()
    plt.show()

## The Theorem: Bias-Variance Tradeoff

The expected prediction error decomposes exactly:

$$E[(\hat{y} - y)^2] = \text{Bias}^2 + \text{Variance} + \sigma^2$$

- **Bias**: how far the model's average prediction is from the truth (too simple = high bias)
- **Variance**: how much the model's predictions change across different training sets (too complex = high variance)
- **$\sigma^2$**: irreducible noise — no model can beat this

Adding parameters **always** reduces bias (more flexibility to match the truth). But it **always** increases variance (more sensitivity to the specific training data). R² on training data only sees bias. It's blind to variance.

In [ ]:
# ── The Destruction Playground: Polynomial degree slider ──

degree_slider = widgets.SelectionSlider(
    options=list(range(1, 16)),
    value=1,
    description='Poly degree:',
    style={'description_width': '90px'},
    layout=widgets.Layout(width='80%'),
    continuous_update=False
)

destroy_output = widgets.Output()
destroy_summary = widgets.Output()

def update_degree(change):
    deg = change['new']
    r = poly_results[deg]

    with destroy_output:
        destroy_output.clear_output(wait=True)
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

        # Left: train vs test R2 curve with current position highlighted
        degrees = list(range(1, 16))
        train_r2s = [poly_results[d]['r2_train'] for d in degrees]
        test_r2s = [poly_results[d]['r2_test'] for d in degrees]

        ax1.plot(degrees, train_r2s, 'o-', color=COLORS['ols'], linewidth=2,
                 markersize=6, label='Training R\u00b2 (always rises)')
        ax1.plot(degrees, test_r2s, 's--', color=COLORS['bias'], linewidth=2,
                 markersize=6, label='Test R\u00b2 (rises then crashes)')
        ax1.axvline(deg, color=COLORS['truth'], linewidth=3, alpha=0.4)
        ax1.plot(deg, r['r2_train'], 'o', color=COLORS['ols'], markersize=14, zorder=5)
        ax1.plot(deg, r['r2_test'], 's', color=COLORS['bias'], markersize=14, zorder=5)
        ax1.axhline(0, color=COLORS['alt'], linestyle=':', alpha=0.5)
        ax1.set_xlabel('Polynomial Degree', fontsize=11)
        ax1.set_ylabel('R\u00b2', fontsize=11)
        ax1.legend(fontsize=9)
        ax1.set_xticks(degrees)

        status = 'UNDERFITTING' if deg <= 2 else ('GOOD' if r['r2_test'] > 0.5 else 'OVERFITTING')
        ax1.set_title(f'Degree {deg}: Train R\u00b2={r["r2_train"]:.3f}, '
                      f'Test R\u00b2={r["r2_test"]:.3f} [{status}]',
                      fontsize=12, fontweight='bold')

        # Right: fitted curve on data
        x_plot = np.linspace(df['ad_spend'].min(), df['ad_spend'].max(), 300).reshape(-1, 1)
        pred = poly_results[deg]['model'].predict(x_plot)

        ax2.scatter(df_train['ad_spend'], df_train['revenue'], alpha=0.2, s=10,
                    color=COLORS['ols'], label='Train')
        ax2.scatter(df_test['ad_spend'], df_test['revenue'], alpha=0.4, s=20,
                    color=COLORS['bias'], marker='^', label='Test')
        ax2.plot(x_plot, pred, color=COLORS['truth'], linewidth=2.5,
                 label=f'Degree {deg} fit')
        ax2.set_xlabel('Ad Spend ($K)', fontsize=11)
        ax2.set_ylabel('Revenue ($K)', fontsize=11)
        ax2.set_title(f'Polynomial Degree {deg} Fit', fontsize=12)
        ax2.legend(fontsize=9)

        plt.tight_layout()
        plt.show()

    with destroy_summary:
        destroy_summary.clear_output(wait=True)
        print(f"Degree: {deg}")
        print(f"Training R\u00b2: {r['r2_train']:.4f}")
        print(f"Test R\u00b2:     {r['r2_test']:.4f}")
        print(f"Gap:         {r['r2_train'] - r['r2_test']:.4f}")
        print(f"CV R\u00b2:      {r['cv_r2']:.4f}")
        if r['r2_test'] < 0:
            print("\n\u2620\ufe0f Predicts worse than the mean!")
        elif r['r2_train'] - r['r2_test'] > 0.1:
            print("\n\u26a0\ufe0f Large train-test gap: overfitting")

degree_slider.observe(update_degree, names='value')
update_degree({'new': 1})

layout = widgets.HBox([
    widgets.VBox([degree_slider, destroy_output], layout=widgets.Layout(width='75%')),
    destroy_summary
])
display(layout)

### Discussion

> *Your team's model has R² = 0.94. The client is impressed. Should you be?*

Think about what questions you'd ask before trusting that number.

In [ ]:
# ── The Repair: Cross-validation + right model family ──

repair_toggle = widgets.ToggleButtons(
    options=['Polynomials Only', 'Add Log-Linear Model'],
    value='Polynomials Only',
    description='Model family:',
    style={'description_width': '100px'},
    button_style=''
)

repair_output = widgets.Output()

def update_repair(change):
    show_log = change['new'] == 'Add Log-Linear Model'

    with repair_output:
        repair_output.clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(11, 5))

        degrees = list(range(1, 16))
        train_r2s = [poly_results[d]['r2_train'] for d in degrees]
        test_r2s = [poly_results[d]['r2_test'] for d in degrees]
        cv_r2s = [poly_results[d]['cv_r2'] for d in degrees]

        ax.plot(degrees, train_r2s, 'o-', color=COLORS['ols'], linewidth=2,
                markersize=6, label='Training R\u00b2')
        ax.plot(degrees, test_r2s, 's--', color=COLORS['bias'], linewidth=2,
                markersize=6, label='Test R\u00b2')
        ax.plot(degrees, cv_r2s, 'D-.', color=COLORS['repair'], linewidth=2,
                markersize=6, label='5-Fold CV R\u00b2')

        if show_log:
            # Add horizontal lines for log-linear model
            ax.axhline(r2_log_train, color=COLORS['ols'], linestyle=':',
                       alpha=0.7, linewidth=1.5)
            ax.axhline(r2_log_test, color=COLORS['truth'], linestyle='-',
                       linewidth=3, alpha=0.8,
                       label=f'Log-linear test R\u00b2 = {r2_log_test:.3f}')
            ax.annotate(f'Log-linear: {r2_log_test:.3f}',
                        xy=(8, r2_log_test), fontsize=12, fontweight='bold',
                        color=COLORS['truth'],
                        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

        ax.axhline(0, color=COLORS['alt'], linestyle=':', alpha=0.5)
        ax.set_xlabel('Polynomial Degree', fontsize=12)
        ax.set_ylabel('R\u00b2', fontsize=12)
        ax.set_xticks(degrees)
        ax.legend(fontsize=10, loc='lower left')

        if show_log:
            ax.set_title('Right model family > right complexity\n'
                         f'Log-linear test R\u00b2 ({r2_log_test:.3f}) beats every polynomial',
                         fontsize=13, fontweight='bold')
        else:
            best_cv_deg = degrees[np.argmax(cv_r2s)]
            ax.set_title(f'CV picks degree {best_cv_deg} '
                         f'(CV R\u00b2={max(cv_r2s):.3f}, test R\u00b2={poly_results[best_cv_deg]["r2_test"]:.3f})',
                         fontsize=13, fontweight='bold')

        plt.tight_layout()
        plt.show()

        if show_log:
            print(f"\nLog-linear model:  Train R\u00b2 = {r2_log_train:.4f},  Test R\u00b2 = {r2_log_test:.4f}")
            print(f"Best polynomial:   Train R\u00b2 = {poly_results[sorted_test[0][0]]['r2_train']:.4f},  "
                  f"Test R\u00b2 = {sorted_test[0][1]['r2_test']:.4f}")
            print(f"\nThe log model wins because the TRUE relationship is logarithmic.")
            print(f"Getting the model family right matters more than tuning complexity.")

repair_toggle.observe(update_repair, names='value')
update_repair({'new': 'Polynomials Only'})

display(widgets.VBox([repair_toggle, repair_output]))

In [ ]:
# ── The Limit: Even CV can overfit with enough searching ──

search_sizes = [1, 2, 5, 10, 20, 50, 100, 200]

# Precompute: for each search breadth, generate random polynomial specs
# and find the best CV R2 among them
limit_cache = {}
rng_search = np.random.default_rng(42)

for n_search in search_sizes:
    best_cv = -np.inf
    best_deg = None
    best_test = None

    # Search over random polynomial specifications
    for _ in range(n_search):
        # Random degree + random feature selection
        deg = rng_search.integers(1, 16)
        if deg in poly_results and poly_results[deg]['cv_r2'] > best_cv:
            best_cv = poly_results[deg]['cv_r2']
            best_deg = deg
            best_test = poly_results[deg]['r2_test']

    limit_cache[n_search] = {
        'best_cv_r2': best_cv,
        'best_deg': best_deg,
        'best_test_r2': best_test
    }

limit_slider = widgets.SelectionSlider(
    options=[(f'{n} models', n) for n in search_sizes],
    value=search_sizes[0],
    description='Models searched:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='80%'),
    continuous_update=False
)

limit_output = widgets.Output()

def update_limit(change):
    n_search = change['new']
    with limit_output:
        limit_output.clear_output(wait=True)

        fig, ax = plt.subplots(figsize=(10, 5))

        sizes = [s for s in search_sizes if s <= n_search]
        cv_scores = [limit_cache[s]['best_cv_r2'] for s in sizes]
        test_scores = [limit_cache[s]['best_test_r2'] for s in sizes]

        ax.plot(sizes, cv_scores, 'D-.', color=COLORS['repair'], linewidth=2,
                markersize=8, label='Best CV R\u00b2 (inflates with search)')
        ax.plot(sizes, test_scores, 's--', color=COLORS['bias'], linewidth=2,
                markersize=8, label='Corresponding test R\u00b2')

        # Gap annotation
        c = limit_cache[n_search]
        gap = c['best_cv_r2'] - c['best_test_r2'] if c['best_test_r2'] is not None else 0

        ax.set_xlabel('Number of Models Searched', fontsize=12)
        ax.set_ylabel('R\u00b2', fontsize=12)
        ax.set_title(f'Searching {n_search} models | Best CV R\u00b2 = {c["best_cv_r2"]:.3f} | '
                     f'CV-Test gap = {gap:.3f}',
                     fontsize=12, fontweight='bold')
        ax.legend(fontsize=10)
        if len(sizes) > 1:
            ax.set_xscale('log')
        plt.tight_layout()
        plt.show()

        if n_search >= 50:
            print("Even cross-validation can overfit if you search too many models.")
            print("The more models you try, the more likely you find one that")
            print("looks good by chance. This is the multiple comparisons problem")
            print("applied to model selection.")

limit_slider.observe(update_limit, names='value')
update_limit({'new': search_sizes[0]})

display(widgets.VBox([limit_slider, limit_output]))

In [ ]:
#@title Real-World Disaster: Long-Term Capital Management { display-mode: "form" }

# ── Sidebar: LTCM ──

story_tab = widgets.HTML(value="""
<div style='padding: 15px; font-size: 13px; line-height: 1.6;'>
<h4>The Story</h4>
<p>Long-Term Capital Management was a hedge fund founded in 1994 by Nobel laureates
Myron Scholes and Robert Merton, along with legendary trader John Meriwether. Their
models were built on decades of historical market data and sophisticated mathematical
finance.</p>

<p>The models worked spectacularly \u2014 for four years. LTCM earned 21%, 43%, 41%, and
17% annual returns. The models fit historical patterns with extraordinary precision.</p>

<p>In August 1998, Russia defaulted on its debt. Market correlations that had been stable
for decades broke down simultaneously. LTCM lost <b>$4.6 billion in less than four months</b>.
The fund's collapse nearly triggered a global financial crisis, requiring a $3.6 billion
bailout organized by the Federal Reserve.</p>

<p>The lesson: their models had overfit to historical patterns. They captured genuine
regularities \u2014 but also noise that happened to be stable during the training period.
When the world changed, the models broke catastrophically.</p>
</div>
""")

math_tab = widgets.HTML(value="""
<div style='padding: 15px; font-size: 13px; line-height: 1.6;'>
<h4>The Math: Bias-Variance at Degree 45</h4>
<p>With 50 observations and a degree-45 polynomial, we have 46 parameters for 50 data points.</p>

<p>The bias-variance decomposition:</p>
<ul>
<li><b>Bias\u00b2 \u2248 0</b>: A degree-45 polynomial can fit almost any smooth function perfectly.
The model is flexible enough to match the truth exactly.</li>
<li><b>Variance \u2192 \u221e</b>: With 46 parameters estimated from 50 observations,
tiny changes in the data cause wild swings in the fitted curve.</li>
<li><b>\u03c3\u00b2</b>: Irreducible noise remains constant regardless of model complexity.</li>
</ul>

<p>Total error = 0 + \u221e + \u03c3\u00b2 = \u221e</p>
<p>Perfect training fit. Disastrous test performance. Exactly what happened to LTCM.</p>
</div>
""")

sim_tab = widgets.Output()

def run_ltcm_sim(btn):
    with sim_tab:
        sim_tab.clear_output(wait=True)
        rng = np.random.default_rng(42)

        # Generate small dataset (50 observations)
        n = 50
        X = rng.uniform(0, 10, n).reshape(-1, 1)
        y_true = 3 * np.sin(X.ravel()) + 2
        y = y_true + rng.normal(0, 1, n)

        # Test set
        X_test_sim = rng.uniform(0, 10, 200).reshape(-1, 1)
        y_test_sim = 3 * np.sin(X_test_sim.ravel()) + 2 + rng.normal(0, 1, 200)

        degrees_sim = [1, 3, 5, 10, 20, 30, 40, 45]
        train_mse = []
        test_mse = []

        for d in degrees_sim:
            if d >= n:
                train_mse.append(0)
                test_mse.append(1e6)
                continue
            pipe = make_pipeline(PolynomialFeatures(d, include_bias=False),
                                LinearRegression())
            pipe.fit(X, y)
            t_pred = pipe.predict(X)
            te_pred = pipe.predict(X_test_sim)
            train_mse.append(np.mean((y - t_pred)**2))
            test_mse.append(np.clip(np.mean((y_test_sim - te_pred)**2), 0, 1000))

        fig, ax = plt.subplots(figsize=(10, 5))
        ax.plot(degrees_sim, train_mse, 'o-', color=COLORS['ols'], linewidth=2,
                markersize=8, label='Training MSE (\u2192 0)')
        ax.plot(degrees_sim, test_mse, 's--', color=COLORS['bias'], linewidth=2,
                markersize=8, label='Test MSE (explodes)')
        ax.set_xlabel('Polynomial Degree', fontsize=12)
        ax.set_ylabel('Mean Squared Error', fontsize=12)
        ax.set_title('50 Observations: Training MSE \u2192 0, Test MSE Explodes',
                     fontsize=13, fontweight='bold')
        ax.legend(fontsize=11)
        ax.set_yscale('symlog', linthresh=1)
        plt.tight_layout()
        plt.show()

        print(f"At degree 45: Training MSE \u2248 0, but test MSE = {test_mse[-1]:.0f}")
        print(f"At degree 3:  Training MSE = {train_mse[1]:.2f}, test MSE = {test_mse[1]:.2f}")
        print(f"\nLTCM's models were like degree-45 polynomials fit to financial history.")
        print(f"Perfect historical fit. Catastrophic real-world failure.")

run_btn = widgets.Button(description='Run Simulation', button_style='primary')
run_btn.on_click(run_ltcm_sim)

sim_content = widgets.VBox([run_btn, sim_tab])

tabs = widgets.Tab(children=[story_tab, math_tab, sim_content])
tabs.set_title(0, 'The Story')
tabs.set_title(1, 'The Math')
tabs.set_title(2, 'The Simulation')

display(tabs)

In [ ]:
# ── Mini Autopsy ──
display(AutopsyReport.lightweight(5))

---

## Bridge

*Your model fits well and predicts well. But does increasing X actually cause Y to increase?*

You've learned to avoid overfitting. Your model generalizes. But generalization doesn't mean causation. A model can perfectly predict revenue from ad spend — and still be completely wrong about what happens if you *change* ad spend.

**Next: Notebook 6 — Why Your Causal Claim Is Wrong**

In [ ]:
# ── Transition Card ──
display(HTML("""
<div style='background: linear-gradient(135deg, #1a1a2e, #16213e); padding: 30px;
            border-radius: 10px; margin: 20px 0; color: white; font-family: sans-serif;'>
    <p style='font-size: 14px; opacity: 0.8; margin: 0;'>NEXT IN THE SERIES</p>
    <h2 style='margin: 10px 0; font-size: 24px;'>Notebook 6: Why Your Causal Claim Is Wrong</h2>
    <p style='font-size: 16px; opacity: 0.9; margin: 10px 0;'>
        If your input and your output cause each other, regression can't tell you
        which direction the effect runs.
    </p>
    <a href='06_causation.ipynb' style='display: inline-block; background: #D4A017;
       color: #1a1a2e; padding: 10px 24px; border-radius: 5px; text-decoration: none;
       font-weight: bold; margin-top: 10px;'>Open Notebook 6 \u2192</a>
</div>
"""))